# Quick Start

Minimum usage for `run_with_cache` with a monitoring engine.


In [1]:
import torch
from transformers import AutoTokenizer
from transformers.models.gpt2_p.modeling_gpt2 import HookedGPT2Model

from monitoring import MonitoringEngine
from monitoring.config import CaptureSchedule, HookSelection, MonitoringConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("gpt2")
token = tokenizer.encode("The future of AI is", return_tensors="pt").to(device)



/home/nengneng/miniconda3/envs/TL/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Init Monitoring Engine with Configuration

### CaptureSchedule

Control when to capture:

```python
CaptureSchedule(
    step_stride=5,           # capture every 5th token
    step_offset=0,           # start from token 0
    request_stride=1,        # capture every request
    capture_prefill=True,    # capture prefill phase
    capture_decode=True,     # capture decode phase
)
```

### HookSelection

Control which hooks to enable:

```python
# Predefined modes
HookSelection(mode='full')       # all 363 hooks
HookSelection(mode='attention')  # attention-related hooks only
HookSelection(mode='mlp')        # MLP-related hooks only

# Custom selection
HookSelection(mode='custom', custom_hooks=['blocks.0.attn.hook_q', 'blocks.0.attn.hook_k'])
```


In [2]:
config = MonitoringConfig(
    hooks=HookSelection(mode="full"),
    schedule=CaptureSchedule(
        step_stride=1,
        step_offset=0,
        warmup_steps=0,
        capture_prefill=True,
        capture_decode=True,
        request_stride=1,
        request_offset=0,
        warmup_requests=0,
    ),
)

engine = MonitoringEngine(async_enabled=device.type == "cuda", config=config)

## Load Hooked Model and Config

In [3]:
hf_hooked_model = HookedGPT2Model.from_pretrained(
    "gpt2",
    attn_implementation="eager",
    torch_dtype=torch.float32,
)
hf_hooked_model.to(device)
hf_hooked_model.eval()
hf_hooked_model.monitoring_engine = engine

init_ms = engine.prepare_for_model(hf_hooked_model)

`torch_dtype` is deprecated! Use `dtype` instead!


In [43]:
engine.begin_request(0)
engine.start_step(phase="prefill")
outputs, cache_dict = hf_hooked_model.run_with_cache(
    token,
    use_cache=True,
    past_key_values=None,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True,
)

engine.end_step()

## decode the next token
next_token = token[:, -1:]
engine.start_step(phase="decode")
outputs, cache_dict = hf_hooked_model.run_with_cache(
    next_token,
    use_cache=True,
    past_key_values=outputs.past_key_values,
    output_hidden_states=True,
    output_attentions=True,
    return_dict=True,
)
engine.end_step()


Get Tensors ready from backend workers

In [44]:
count = 0
for name, fut in cache_dict.items():
    if fut is None:
        continue
    cache_dict[name] = fut.result()
    if cache_dict[name] is not None:
        count += 1

assert count == len(cache_dict.items()), "Some are missing"
print("All tensor are ready")

        

All tensor are ready


In [45]:
cache_dict.keys()
print(cache_dict['blocks.0.hook_resid_pre'].shape)

torch.Size([1, 1, 768])


In [46]:
engine.clear_completed_results()
cache_dict.clear()
